# 05 Technical Agent

Generate a technical chart for a ticker and use a multimodal Azure OpenAI chat deployment to answer questions about the chart. The notebook stays thin: market-data loading, indicator calculation, charting, and multimodal message construction live in reusable project modules.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import Image, display

from market_analyst.config.settings import load_settings
from market_analyst.providers.market_data import fetch_price_history
from market_analyst.services.agents.technical import (
    DEFAULT_TECHNICAL_QUESTION,
    analyze_technical_chart,
    build_multimodal_chart_message,
)
from market_analyst.services.charting import (
    add_technical_indicators,
    generate_technical_chart,
    summarize_chart_artifact,
)
from market_analyst.telemetry import configure_notebook_logging
from market_analyst.types.technical import TechnicalAnalysisRequest

logger = configure_notebook_logging(run_name="05_technical_agent")
settings = load_settings()
settings.require_chat_model()

print("Project root:", PROJECT_ROOT)
print("Chat deployment:", settings.azure_openai_chat_deployment)

## Run Configuration

Use a Yahoo Finance-compatible ticker. `QUESTION` is sent with the generated chart image to the multimodal model.

In [ ]:
REQUEST = TechnicalAnalysisRequest(
    ticker="MSFT",
    period="6mo",
    interval="1d",
    question=DEFAULT_TECHNICAL_QUESTION,
)

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "technical_charts"
print(REQUEST)

## Fetch Prices And Generate Chart

In [ ]:
prices = fetch_price_history(REQUEST.ticker, period=REQUEST.period, interval=REQUEST.interval)
prices_with_indicators = add_technical_indicators(prices)
artifact = generate_technical_chart(REQUEST.ticker, prices_with_indicators, output_dir=OUTPUT_DIR)

print(summarize_chart_artifact(artifact))
display(Image(filename=str(artifact.chart_path)))

## Inspect Multimodal Payload

This confirms that the agent call will include one text part and one chart image part.

In [ ]:
message = build_multimodal_chart_message(artifact=artifact, question=REQUEST.question)
print("Payload parts:", [part["type"] for part in message.content])
print(message.content[0]["text"][:900])

## Ask The Technical Agent

In [ ]:
answer = analyze_technical_chart(settings, artifact=artifact, question=REQUEST.question)
print(answer)

## Validation

In [ ]:
assert artifact.chart_path.exists(), "Technical chart image should exist."
assert artifact.observation_count > 20, "Technical analysis needs enough price observations."
assert str(answer).strip(), "Technical agent answer should not be empty."

print("Technical agent notebook validation passed.")